# Imports

In [1]:
import optuna
import pickle

import numpy as np
import pandas as pd

from tqdm import tqdm

from sklearn.metrics import balanced_accuracy_score
from sklearn.model_selection import StratifiedKFold, cross_val_predict

/home/junior/Documentos/GitHub/kaggle-competition-predicting-student-health/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Utils

In [2]:
def load_pickle(file_path):
    with open(file_path, 'rb') as file:
        return pickle.load(file)

# Loading Datasets

In [4]:
X_train = pd.read_parquet('../data/processed/X_train_stacking_layer_one_model_60_trials_data_type_raw_target_encoding.parquet')
y_train = pd.read_parquet('../data/interim/y_train.parquet')

X_test = pd.read_parquet('../data/processed/X_test_stacking_layer_one_model_60_trials_data_type_raw_target_encoding.parquet')

In [5]:
X_train.head()

,lgbm_0,lgbm_1,lgbm_2,xgb_0,xgb_1,xgb_2
0,0.003278,0.000834,0.995889,0.005375,0.001396,0.993230
1,0.997061,0.000223,0.002716,0.995312,0.000386,0.004302
2,0.003584,0.000622,0.995794,0.003329,0.000530,0.996142
3,0.003934,0.003009,0.993058,0.004901,0.001941,0.993158
4,0.995190,0.000011,0.004800,0.993787,0.000018,0.006195


In [6]:
X_test.head()

,lgbm_0,lgbm_1,lgbm_2,xgb_0,xgb_1,xgb_2
0,0.012261,0.002968,0.984771,0.011874,0.003576,0.984551
1,0.405696,0.002057,0.592248,0.459639,0.002737,0.537624
2,0.999251,0.000597,0.000152,0.998100,0.001247,0.000653
3,0.981066,0.000008,0.018926,0.987928,0.000017,0.012055
4,0.012981,0.002376,0.984643,0.009526,0.002050,0.988424


# Machine Learning

In [7]:
label_encoder = load_pickle('../artifacts/label_encoder.pkl')

In [8]:
def objective(trial, X, y):

    w0 = trial.suggest_float('weight_class_0', 0.001, 10.0, log=True)
    w1 = trial.suggest_float('weight_class_1', 0.001, 10.0, log=True)
    w2 = trial.suggest_float('weight_class_2', 0.001, 10.0, log=True)
    weights = np.array([w0, w1, w2])

    proba = X.loc[:, ['xgb_0', 'xgb_1', 'xgb_2']].to_numpy()
    
    weighted_probas = proba * weights
    pred = np.argmax(weighted_probas, axis=1)
    
    score = balanced_accuracy_score(y, pred)

    return score


study = optuna.create_study(
    direction="maximize", 
    sampler=optuna.samplers.TPESampler(seed=42), 
    pruner=optuna.pruners.MedianPruner(n_warmup_steps=2)
)

study.optimize(
    lambda trial: objective(trial, X_train, y_train.health_condition), 
    n_trials=500, 
    n_jobs=-1, 
    show_progress_bar=True
)

print("\nBest trial score:")
print(study.best_trial.value)

print("\nBest params:")
print(study.best_trial.params)

[I 2026-07-13 11:29:19,474] A new study created in memory with name: no-name-de9356d6-65b2-4934-aeda-ad27eb5af3ab
Best trial: 6. Best value: 0.949769:   2%|███▎                                                                                                                                    | 12/500 [00:00<00:12, 38.28it/s]

[I 2026-07-13 11:29:19,633] Trial 4 finished with value: 0.5840954108970536 and parameters: {'weight_class_0': 9.36612653802759, 'weight_class_1': 0.0020783866337358223, 'weight_class_2': 2.066964605179856}. Best is trial 4 with value: 0.5840954108970536.
[I 2026-07-13 11:29:19,636] Trial 10 finished with value: 0.609288813387205 and parameters: {'weight_class_0': 3.334438667662059, 'weight_class_1': 0.11209246625021807, 'weight_class_2': 0.007803132439923141}. Best is trial 10 with value: 0.609288813387205.
[I 2026-07-13 11:29:19,640] Trial 5 finished with value: 0.6761491590029621 and parameters: {'weight_class_0': 0.1609400218146566, 'weight_class_1': 0.0011556652010483528, 'weight_class_2': 0.016267395904175082}. Best is trial 5 with value: 0.6761491590029621.
[I 2026-07-13 11:29:19,649] Trial 6 finished with value: 0.9497686830141454 and parameters: {'weight_class_0': 0.635411096028422, 'weight_class_1': 9.440117768949326, 'weight_class_2': 7.042410518539365}. Best is trial 6 with

Best trial: 6. Best value: 0.949769:   5%|██████▊                                                                                                                                 | 25/500 [00:00<00:09, 49.45it/s]

[I 2026-07-13 11:29:19,811] Trial 13 finished with value: 0.9481969073633304 and parameters: {'weight_class_0': 0.2853738559208835, 'weight_class_1': 9.455553622131278, 'weight_class_2': 6.971465635537982}. Best is trial 6 with value: 0.9497686830141454.
[I 2026-07-13 11:29:19,818] Trial 14 finished with value: 0.9475582634871685 and parameters: {'weight_class_0': 0.24889988177815953, 'weight_class_1': 1.2025852361647902, 'weight_class_2': 2.192296987316923}. Best is trial 6 with value: 0.9497686830141454.
[I 2026-07-13 11:29:19,824] Trial 15 finished with value: 0.9456926216110976 and parameters: {'weight_class_0': 0.24611777092063994, 'weight_class_1': 1.212906284204062, 'weight_class_2': 7.9401665505443555}. Best is trial 6 with value: 0.9497686830141454.
[I 2026-07-13 11:29:19,859] Trial 17 finished with value: 0.9456290078091786 and parameters: {'weight_class_0': 0.2606593206207706, 'weight_class_1': 1.37883415382351, 'weight_class_2': 9.857470622569783}. Best is trial 6 with valu

Best trial: 6. Best value: 0.949769:   8%|██████████▎                                                                                                                             | 38/500 [00:00<00:08, 54.28it/s]

...

[I 2026-07-13 11:29:27,918] Trial 460 finished with value: 0.7079300097660419 and parameters: {'weight_class_0': 0.6565921551691782, 'weight_class_1': 6.683547606780894, 'weight_class_2': 0.004420292220121945}. Best is trial 284 with value: 0.9498687991528469.
[I 2026-07-13 11:29:27,948] Trial 463 finished with value: 0.9420572461078396 and parameters: {'weight_class_0': 1.80325433456134, 'weight_class_1': 6.896754478239845, 'weight_class_2': 6.624761555289973}. Best is trial 284 with value: 0.9498687991528469.
[I 2026-07-13 11:29:27,950] Trial 462 finished with value: 0.7205986018390043 and parameters: {'weight_class_0': 0.013315106439532012, 'weight_class_1': 6.77797847436737, 'weight_class_2': 6.4736524620245515}. Best is trial 284 with value: 0.9498687991528469.
[I 2026-07-13 11:29:27,990] Trial 464 finished with value: 0.9497727950721825 and parameters: {'weight_class_0': 0.5762629902076984, 'weight_class_1': 7.4957811530655905, 'weight_class_2': 6.429701499751161}. Best is trial 

Best trial: 284. Best value: 0.949869:  97%|████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████▍    | 483/500 [00:08<00:00, 54.90it/s]

[I 2026-07-13 11:29:28,122] Trial 473 finished with value: 0.9498143725149992 and parameters: {'weight_class_0': 0.5295771535932572, 'weight_class_1': 8.308358975273839, 'weight_class_2': 5.4853648523688765}. Best is trial 284 with value: 0.9498687991528469.
[I 2026-07-13 11:29:28,151] Trial 474 finished with value: 0.9498413878866462 and parameters: {'weight_class_0': 0.5498598835683911, 'weight_class_1': 8.121524388505092, 'weight_class_2': 5.593023312906279}. Best is trial 284 with value: 0.9498687991528469.
[I 2026-07-13 11:29:28,186] Trial 475 finished with value: 0.9492043841034916 and parameters: {'weight_class_0': 0.42409320058853667, 'weight_class_1': 8.195357794655248, 'weight_class_2': 8.076395942822257}. Best is trial 284 with value: 0.9498687991528469.
[I 2026-07-13 11:29:28,206] Trial 477 finished with value: 0.9491441750445812 and parameters: {'weight_class_0': 0.4200040246544635, 'weight_class_1': 8.529047408444882, 'weight_class_2': 8.377332208963681}. Best is trial 28

Best trial: 284. Best value: 0.949869: 100%|█████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 500/500 [00:09<00:00, 55.03it/s]

[I 2026-07-13 11:29:28,341] Trial 482 finished with value: 0.9497526844673736 and parameters: {'weight_class_0': 0.4273900718856847, 'weight_class_1': 8.446345585837355, 'weight_class_2': 4.368681688680055}. Best is trial 284 with value: 0.9498687991528469.
[I 2026-07-13 11:29:28,357] Trial 485 finished with value: 0.6729840976204837 and parameters: {'weight_class_0': 0.4304358487352327, 'weight_class_1': 8.282016667793059, 'weight_class_2': 0.0013876641208885733}. Best is trial 284 with value: 0.9498687991528469.
[I 2026-07-13 11:29:28,386] Trial 487 finished with value: 0.9496373961846108 and parameters: {'weight_class_0': 0.3183574194162161, 'weight_class_1': 5.5243100636438545, 'weight_class_2': 4.4092521662532285}. Best is trial 284 with value: 0.9498687991528469.
[I 2026-07-13 11:29:28,392] Trial 486 finished with value: 0.9485625838990419 and parameters: {'weight_class_0': 0.7868854889859811, 'weight_class_1': 8.52869403368548, 'weight_class_2': 4.356933999614687}. Best is trial

In [9]:
weights = np.array([study.best_params['weight_class_0'], study.best_params['weight_class_1'], study.best_params['weight_class_2']])
weighted_probas = X_test.loc[:, ['xgb_0', 'xgb_1', 'xgb_2']].to_numpy() * weights

pred = np.argmax(weighted_probas, axis=1)

In [10]:
sub_labels = label_encoder.inverse_transform(pred)

# Submission

In [11]:
submission = pd.read_csv('../data/submission/sample_submission.csv')
submission['health_condition'] = sub_labels

submission.to_csv('../data/submission/submission_from_5.1_xgboost_submission.csv', index=False)

In [12]:
submission.head()

,id,health_condition
0,690088,unhealthy
1,690089,unhealthy
2,690090,at-risk
3,690091,at-risk
4,690092,unhealthy
